# Setup

In [33]:
# Optional autoreload while developing locally

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [34]:
# to load the virtual environment
## & "$HOME\nlp_owlapy_env_py311\Scripts\Activate.ps1"

In [35]:

# 0) Local-only setup and base diagnostics
import os
import sys
import platform
from pathlib import Path

print("Python:", sys.version)
print("Platform:", platform.platform())

PROJECT_DIR = Path.cwd()



Python: 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
Platform: Windows-10-10.0.26200-SP0


In [36]:

# 0.3) Shared device diagnostics
import torch

print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU count:", torch.cuda.device_count())
    print("GPU 0:", torch.cuda.get_device_name(0))
    DEVICE = "cuda:0"
else:
    DEVICE = "cpu"
print("Using DEVICE =", DEVICE)


torch: 2.11.0+cu128
CUDA available: True
GPU count: 1
GPU 0: NVIDIA RTX A2000 8GB Laptop GPU
Using DEVICE = cuda:0


# Imports

In [37]:
import pickle
import re

import pandas as pd
from spacy.tokens import Doc

from dataclasses import dataclass, asdict
from itertools import product

from tokenizer import create_tokenizer, ensure_booknlp_extensions, tokens_to_dicts
from doc_chunker import create_chunker
from runtime_config import (
    RUNTIME_PROFILE,
    DEVICE,
    CHUNK_SIZE,
    OVERLAP_SENTENCES,
    MAX_EXPANDED_CHUNK_TOKENS,
)

In [38]:
from coreference.coreference_sub_orchestrator import (
    run_coreference_resolution,
    print_non_singleton_coref_clusters,
)

from coreference.coref_schema import require_coref_layer

In [39]:
from ocean.ocean_probability_scoring import (
    ContextConfig,
    MentionRenderingConfig,
    OceanScoringConfig,
    OceanWeightConfig,
    DirectNLIConfig,
    OceanProbabilityExportConfig,
    export_ocean_probability_csvs,
)

from ocean.ocean_annotator import annotate_doc_with_ocean_folder
from ocean.ocean_schema import (
    require_ocean_layer,
    register_spacy_ocean_extension
)

In [40]:
from ontology_graph_builder import build_networkx_graph_from_ttl, ontology_label_tree

from cluster_typing.ontology_probability_scoring import (
    OntologyTraversalConfig,
    OntologyScoringConfig,
    OntologyMentionWeightConfig,
    OntologyEvidenceExportConfig,
    export_ontology_evidence_jsonls,
)

from cluster_typing.ontology_annotator import (
    OntologyAnnotationConfig,
    annotate_doc_with_ontology_folder,
)

from cluster_typing.ontology_schema import (
    register_spacy_ontology_extension,
    require_ontology_layer,
)

In [41]:
from ontology.ontology_managment import (
    assert_cluster_relation,
    assert_cluster_type,
    assert_cluster_value,
    load_tbox,
    save_ontology,
    build_relation_catalog,
    build_relation_router,
    # assert_cluster_object_property,
)

In [42]:
from relationship_extraction.relation_schema import (
    register_spacy_relation_extension,
    require_relation_layer,
)

from relationship_extraction.extract_relation_candidates import (
    extract_relation_candidates,
    export_routed_relation_candidates_jsonl,
)

from relationship_extraction.align_relation_assignments import (
    RelationNLIConfig,
    load_relation_nli_selector,
    export_relation_assignments_csv,
)

from relationship_extraction.aggregate_cluster_assertions import (
    RelationAggregationConfig,
    export_cluster_assertions_csv,
)

from relationship_extraction.annotate_relation_layer import (
    annotate_relation_layer_from_files,
)

# Functions

In [43]:
# text loading + preprocessing
def load_txt(path):
    path = Path(path)

    try:
        return path.read_text(encoding="utf-8")
    except UnicodeDecodeError:
        return path.read_text(encoding="latin-1")

In [44]:
import re

def preprocess_for_NER(text):
    # Normalize Windows/Mac newlines
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    # Remove possible invisible BOM
    text = text.replace("\ufeff", "")

    # Normalize curly apostrophes/quotes
    text = text.replace("’", "'")
    text = text.replace("‘", "'")
    text = text.replace("“", '"').replace("”", '"')

    # Remove standalone page numbers, e.g.
    # This removes only lines that contain digits plus optional whitespace.
    text = re.sub(r"(?m)^[ \t]*\d+[ \t]*$", "", text)

    # Replace line breaks inside paragraphs with spaces,
    # but preserve paragraph boundaries
    paragraphs = text.split("\n\n")
    paragraphs = [
        re.sub(r"\s*\n\s*", " ", paragraph).strip()
        for paragraph in paragraphs
        if paragraph.strip()
    ]

    text = "\n\n".join(paragraphs)

    chapter_re = re.compile(
        r"(?im)^\s*chapter\s+(?:[ivxlcdm]+|\d+)\b[^\n]*\n*"
    )

    text = chapter_re.sub("", text)

    # Normalize multiple spaces
    text = re.sub(r"[ \t]+", " ", text)

    return text.strip()

In [45]:

def resolve_text_path(
    *,
    project_dir: Path,
    filename: str,
    local_path: str | Path,
) -> Path:
    """Resolve the local input text path."""
    text_path = Path(local_path)
    if text_path.exists():
        return text_path

    fallback_path = project_dir / filename
    if fallback_path.exists():
        return fallback_path

    raise FileNotFoundError(
        "Could not resolve a local text file. "
        f"Tried LOCAL_TEXT_PATH={text_path} and PROJECT_DIR / {filename!r} = {fallback_path}."
    )


In [46]:
def save_doc(doc, path: str | Path) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("wb") as f:
        pickle.dump(doc, f)
    return path


def load_doc(path: str | Path):
    # BookNLP/tokenizer extensions must be registered before unpickling the Doc.
    # The global coreference extension is registered inside the coreference
    # sub-orchestrator immediately before final annotation.
    ensure_booknlp_extensions()
    register_spacy_ontology_extension()
    register_spacy_ocean_extension()
    register_spacy_relation_extension()
    
    path = Path(path)
    with path.open("rb") as f:
        return pickle.load(f)


# Config

## I/O config

In [47]:
# Requested local file
TEXT_FILENAME = "oz_full.txt"
LOCAL_TEXT_PATH = Path(f"./resources/books/{TEXT_FILENAME}")

TEXT_PATH = resolve_text_path(
    project_dir=PROJECT_DIR,
    filename=TEXT_FILENAME,
    local_path=LOCAL_TEXT_PATH,
)

OUTPUT_ROOT = PROJECT_DIR / f"outputs_[{TEXT_FILENAME}]"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
ONTOLOGY_ROOT = PROJECT_DIR / "resources/ontologies"

TOKENIZED_DOC_PATH = OUTPUT_ROOT / "tokenized_doc.pkl"
COREF_DOC_PATH = OUTPUT_ROOT / "coref_doc.pkl"
ONTOLOGY_TYPED_DOC_PATH = OUTPUT_ROOT / "ontology_typed_doc.pkl"
OCEAN_SCORED_DOC_PATH = OUTPUT_ROOT / "ocean_scored_doc.pkl"

ONTOLOGY_TTL_PATH = ONTOLOGY_ROOT/ "narrative_entity_ontology_clean.ttl"

RELATION_OUTPUT_DIR = OUTPUT_ROOT / "relations"
RELATION_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ROUTED_RELATION_CANDIDATES_PATH = RELATION_OUTPUT_DIR / "routed_relation_candidates.jsonl"
RELATION_ASSIGNMENTS_PATH = RELATION_OUTPUT_DIR / "relation_assignments.csv"
CLUSTER_ASSERTIONS_PATH = RELATION_OUTPUT_DIR / "cluster_assertions.csv"
RELATION_DOC_PATH = OUTPUT_ROOT / "relation_doc.pkl"
POPULATED_ONTOLOGY_PATH = OUTPUT_ROOT / "populated_ontology.ttl"

print("TEXT_PATH =", TEXT_PATH)
print("OUTPUT_ROOT =", OUTPUT_ROOT)
print("TOKENIZED_DOC_PATH =", TOKENIZED_DOC_PATH)
print("COREF_DOC_PATH =", COREF_DOC_PATH)
print("ONTOLOGY_TYPED_DOC_PATH =", ONTOLOGY_TYPED_DOC_PATH)
print("OCEAN_SCORED_DOC_PATH =", OCEAN_SCORED_DOC_PATH)
print("ONTOLOGY_TTL_PATH =", ONTOLOGY_TTL_PATH)


TEXT_PATH = resources\books\oz_full.txt
OUTPUT_ROOT = c:\Users\Bobby\Desktop\real_shit\NLP_character_graph_pipeline\outputs_[oz_full.txt]
TOKENIZED_DOC_PATH = c:\Users\Bobby\Desktop\real_shit\NLP_character_graph_pipeline\outputs_[oz_full.txt]\tokenized_doc.pkl
COREF_DOC_PATH = c:\Users\Bobby\Desktop\real_shit\NLP_character_graph_pipeline\outputs_[oz_full.txt]\coref_doc.pkl
ONTOLOGY_TYPED_DOC_PATH = c:\Users\Bobby\Desktop\real_shit\NLP_character_graph_pipeline\outputs_[oz_full.txt]\ontology_typed_doc.pkl
OCEAN_SCORED_DOC_PATH = c:\Users\Bobby\Desktop\real_shit\NLP_character_graph_pipeline\outputs_[oz_full.txt]\ocean_scored_doc.pkl
ONTOLOGY_TTL_PATH = c:\Users\Bobby\Desktop\real_shit\NLP_character_graph_pipeline\resources\ontologies\narrative_entity_ontology_clean.ttl


In [48]:
GLOBAL_COREF_OUTPUT_DIR = OUTPUT_ROOT / "global_coref"
GLOBAL_COREF_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("GLOBAL_COREF_OUTPUT_DIR =", GLOBAL_COREF_OUTPUT_DIR)

GLOBAL_COREF_OUTPUT_DIR = c:\Users\Bobby\Desktop\real_shit\NLP_character_graph_pipeline\outputs_[oz_full.txt]\global_coref


## Runtime and pipeline config

In [49]:
# Pipeline configuration
SPACY_MODEL = "en_core_web_sm"
TOKENIZER_DISABLE = ("ner",)

RELATION_PAIR_BATCH_SIZE = 64
RELATION_OVERWRITE_STAGE_1 = False
RELATION_OVERWRITE_STAGE_2 = False
RELATION_RESUME_STAGE_2 = True
RELATION_OVERWRITE_STAGE_3 = True

print("RUNTIME_PROFILE =", RUNTIME_PROFILE)
print("DEVICE =", DEVICE)
print("CHUNK_SIZE(expanded) =", CHUNK_SIZE)
print("OVERLAP_SENTENCES =", OVERLAP_SENTENCES)
print("MAX_EXPANDED_CHUNK_TOKENS =", MAX_EXPANDED_CHUNK_TOKENS)
print("GLOBAL_COREF_OUTPUT_DIR =", GLOBAL_COREF_OUTPUT_DIR)


RUNTIME_PROFILE = {'kind': 'local_cuda', 'env': 'Local', 'cuda_available': True, 'gpu_count': 1, 'gpu_name': 'NVIDIA RTX A2000 8GB Laptop GPU', 'device': 'cuda:0', 'cpu_load_first': True, 'precision_policy': 'auto', 'p100_fallback_to_float32': False, 'default_chunk_size': 6000, 'default_overlap_sentences': 60}
DEVICE = cuda:0
CHUNK_SIZE(expanded) = 6000
OVERLAP_SENTENCES = 60
MAX_EXPANDED_CHUNK_TOKENS = 6000
GLOBAL_COREF_OUTPUT_DIR = c:\Users\Bobby\Desktop\real_shit\NLP_character_graph_pipeline\outputs_[oz_full.txt]\global_coref


# Main

### Tokenization

In [50]:
# Load or create the stable tokenized Doc artifact.
if TOKENIZED_DOC_PATH.exists():
    print(f"[doc] Loading tokenized Doc from {TOKENIZED_DOC_PATH}")
    doc = load_doc(TOKENIZED_DOC_PATH)
else:
    print("[doc] Tokenized Doc not found; creating it from raw text.")
    raw_text = load_txt(TEXT_PATH)
    text = preprocess_for_NER(raw_text)
    tokenizer = create_tokenizer(
        SPACY_MODEL,
        disable=TOKENIZER_DISABLE,
        verbose=True,
    )
    doc = tokenizer.tokenize(text)
    save_doc(doc, TOKENIZED_DOC_PATH)
    print(f"[doc] Saved tokenized Doc to {TOKENIZED_DOC_PATH}")

print("Doc tokens:", len(doc))
print("Doc sentences:", sum(1 for _ in doc.sents))

[doc] Loading tokenized Doc from c:\Users\Bobby\Desktop\real_shit\NLP_character_graph_pipeline\outputs_[oz_full.txt]\tokenized_doc.pkl
Doc tokens: 48045
Doc sentences: 1943


### Chunking

In [ ]:
chunker = create_chunker(
    chunk_size=CHUNK_SIZE,
    overlap_sentences=OVERLAP_SENTENCES,
    max_expanded_chunk_tokens=MAX_EXPANDED_CHUNK_TOKENS,
)
chunk_plan = chunker.plan(doc)

### Ontology building


In [ ]:
onto, ontology_graph = load_tbox(ONTOLOGY_TTL_PATH, require_property_descriptions=True)

print(ontology_graph.number_of_nodes(), "classes")
print(ontology_graph.number_of_edges(), "subclass edges")

relation_catalog = build_relation_catalog(
    onto=onto,
    ontology_path=ONTOLOGY_TTL_PATH,
)

relation_router = build_relation_router(
    class_graph=ontology_graph,
    relation_catalog=relation_catalog,
)

print("Object properties:", len(relation_catalog))

25 classes
22 subclass edges
Object properties: 16


## Node extraction

### Coreference clusters extraction

In [ ]:
if COREF_DOC_PATH.exists():
    print(f"[coref-annotation] Loading coref-annotated Doc from {COREF_DOC_PATH}")
    doc = load_doc(COREF_DOC_PATH)
else:
    print("[coref-annotation] Annotated Doc not found; calling annotator sub-orchestrator.")
    doc = run_coreference_resolution(
        doc=doc,
        chunker=chunker,
        chunk_plan=chunk_plan,
        document_id=TEXT_PATH.stem,
        output_dir=GLOBAL_COREF_OUTPUT_DIR,
    )
    save_doc(doc, COREF_DOC_PATH)
    print("[coref-annotation] Saved annotated Doc to:", COREF_DOC_PATH)
    
    
if not Doc.has_extension("coref_layer"):
    Doc.set_extension("coref_layer", default=None)

[coref-annotation] Loading coref-annotated Doc from c:\Users\Bobby\Desktop\real_shit\NLP_character_graph_pipeline\outputs_[oz_full.txt]\coref_doc.pkl


In [ ]:
print("[coref-annotation] Summary:")
print(doc._.coref_layer.summary())
print_non_singleton_coref_clusters(doc)

[coref-annotation] Summary:
{'n_mentions': 9460, 'n_clusters': 566, 'n_indexed_tokens': 11462, 'n_indexed_heads': 7346, 'n_indexed_spans': 7375}
Non-singleton clusters: 199

cluster_id=8 | canonical_name='Dorothy' | n_mentions=1900

cluster_id=97 | canonical_name='the Scarecrow' | n_mentions=1032

cluster_id=153 | canonical_name='the Lion' | n_mentions=837

cluster_id=48 | canonical_name='Oz' | n_mentions=801

cluster_id=111 | canonical_name='the travelers' | n_mentions=598

cluster_id=76 | canonical_name='the Wicked Witch' | n_mentions=357

cluster_id=226 | canonical_name='the Tin Woodman' | n_mentions=258

cluster_id=20 | canonical_name='Toto' | n_mentions=179

cluster_id=60 | canonical_name='the Emerald City' | n_mentions=171

cluster_id=41 | canonical_name='her friends' | n_mentions=140

cluster_id=4 | canonical_name='Aunt Em' | n_mentions=117

cluster_id=252 | canonical_name='the people' | n_mentions=99

cluster_id=67 | canonical_name='the Winkies' | n_mentions=94

cluster_id=1 | 

### Ontology cluster typing

In [ ]:
ontology_n_mentions = None  # None = type ALL mentions for each requested cluster
ontology_random_seed = 67

In [ ]:
if ONTOLOGY_TYPED_DOC_PATH.exists():
    print(f"[ontology typing] Loading ontology-typed Doc from {ONTOLOGY_TYPED_DOC_PATH}")
    doc = load_doc(ONTOLOGY_TYPED_DOC_PATH)
else:
    if not ONTOLOGY_TTL_PATH.exists():
        raise FileNotFoundError(
            f"Ontology TTL not found: {ONTOLOGY_TTL_PATH}. "
            "Create a networkx.DiGraph by any external method, or place the TTL here."
        )

    jsonl_paths_by_cluster_id = export_ontology_evidence_jsonls(
        doc,
        ontology_graph,
        OntologyEvidenceExportConfig(
            n_mentions_per_cluster=ontology_n_mentions,
            output_root=OUTPUT_ROOT,

            context_config=ContextConfig(
                n_sentences_before=0,
                n_sentences_after=0,
                mark_mention=True,
                deduplicate=False,
            ),
            rendering_config=MentionRenderingConfig(
                canonicalize_simple_mentions=True,
                keep_first_second_person=True,
            ),
            traversal_config=OntologyTraversalConfig(
                skip_single_root=True,
                include_stay_option=True,
                force_leaf=False,
            ),
            scoring_config=OntologyScoringConfig(
                subject_aware=True,
            ),
            mention_weight_config=OntologyMentionWeightConfig(),
            nli_config=DirectNLIConfig(
                pair_batch_size=64,
            ),

            chunk_size=16,

            overwrite_jsonl=False,
            resume_from_jsonl=True,

            random_seed=ontology_random_seed,
            sort_sample_by_cluster_order=True,
            print_progress=True,
        ),
    )

    print(jsonl_paths_by_cluster_id)

    n_part = "all" if ontology_n_mentions is None else str(ontology_n_mentions)
    ontology_folder = OUTPUT_ROOT / "ontology_typing" / n_part

    doc = annotate_doc_with_ontology_folder(
        doc,
        ontology_graph,
        ontology_folder,
        config=OntologyAnnotationConfig(
            use_mention_weight=True,
        ),
    )
    save_doc(doc, ONTOLOGY_TYPED_DOC_PATH)
    print("\n\n\n[ontology typing] Saved annotated Doc to:", ONTOLOGY_TYPED_DOC_PATH)

ontology_layer = require_ontology_layer(doc)
print(ontology_layer.summary())

[ontology typing] Loading ontology-typed Doc from c:\Users\Bobby\Desktop\real_shit\NLP_character_graph_pipeline\outputs_[oz_full.txt]\ontology_typed_doc.pkl
{'n_ontology_clusters': 566, 'n_ontology_classes_used': 15}


In [ ]:
coref = doc._.coref_layer
ontology_layer = doc._.ontology_layer

rows = []

for cluster_id, cluster in sorted(coref.clusters.items()):
    annotation = ontology_layer.cluster(cluster_id)

    rows.append(
        {
            "cluster_id": cluster_id,
            "canonical_name": cluster.canonical_name,
            "semantic_type": cluster.semantic_type,
            "n_mentions": len(cluster.mention_ids),
            "ontology_class_label": annotation.class_label,
            "ontology_class_human_readable_label": annotation.class_human_readable_label,
        }
    )

ontology_clusters_df = pd.DataFrame(rows)

ontology_clusters_df

,cluster_id,canonical_name,semantic_type,n_mentions,ontology_class_label,ontology_class_human_readable_label
0,0,the great Kansas prairies,UNKNOWN,1,Site,Site
1,1,Kansas,UNKNOWN,93,Region,Region
2,2,Uncle Henry,UNKNOWN,49,Human character,Human Character
3,3,a farmer,UNKNOWN,29,Human character,Human Character
4,4,Aunt Em,UNKNOWN,117,Human character,Human Character
...,...,...,...,...,...,...
561,561,no one,UNKNOWN,1,Narrative entity,Narrative Entity
562,562,the people,UNKNOWN,1,Belief or value,Belief Or Value
563,563,the people,UNKNOWN,1,Site,Site
564,564,her loving comrades,UNKNOWN,1,Belief or value,Belief Or Value


### OCEAN scoring

In [ ]:
def cluster_ids_of_class_or_subclass(doc, coref_layer, ontology_layer, class_label):
    coref_cluster_ids = set(coref_layer.clusters)

    return [
        cluster_id
        for cluster_id in ontology_layer.cluster_ids_under(class_label)
        if cluster_id in coref_cluster_ids
    ]

In [ ]:

coref = doc._.coref_layer
ontology_layer = doc._.ontology_layer

cluster_ids = cluster_ids_of_class_or_subclass(
    doc,
    coref,
    ontology_layer,
    "Character",
)

for id in sorted(cluster_ids):
    print(id, end=', ')

2, 3, 4, 8, 19, 20, 21, 33, 45, 48, 54, 56, 63, 67, 71, 76, 80, 90, 97, 102, 103, 106, 128, 136, 143, 146, 151, 153, 157, 158, 160, 161, 167, 178, 195, 201, 207, 211, 213, 221, 222, 226, 238, 243, 244, 267, 287, 288, 295, 299, 301, 305, 307, 312, 320, 322, 323, 333, 334, 338, 342, 354, 358, 359, 366, 372, 379, 383, 387, 388, 391, 392, 393, 397, 409, 416, 435, 439, 455, 456, 458, 469, 470, 472, 475, 480, 484, 489, 498, 499, 500, 502, 510, 511, 521, 528, 529, 533, 534, 539, 540, 

In [ ]:
n_mentions = None  # None = score ALL mentions for each requested cluster
random_seed = 67

In [ ]:
if OCEAN_SCORED_DOC_PATH.exists():
    print(f"[OCEAN scoring] Loading ocean_scoring-annotated Doc from {OCEAN_SCORED_DOC_PATH}")
    doc = load_doc(OCEAN_SCORED_DOC_PATH)
else:

    csv_paths_by_cluster_id = export_ocean_probability_csvs(
    doc,
    OceanProbabilityExportConfig(
        cluster_ids=cluster_ids,
        n_mentions_per_cluster=n_mentions,
        output_root="./outputs",

        context_config=ContextConfig(
            n_sentences_before=0,
            n_sentences_after=0,
            mark_mention=True,
            deduplicate=False,
        ),
        rendering_config=MentionRenderingConfig(
            canonicalize_simple_mentions=True,
            keep_first_second_person=True,
        ),
        scoring_config=OceanScoringConfig(
            subject_aware=True,
        ),
        weight_config=OceanWeightConfig(),
        nli_config=DirectNLIConfig(
            pair_batch_size=64, # on stronger hardware can be safely set to 64
        ),

        chunk_size=64,

        overwrite_csv=False,
        resume_from_csv=True,
        use_sqlite_cache=True,

        random_seed=random_seed,
        sort_sample_by_cluster_order=True,
        return_dataframes=False,
        print_progress=True,
    ),
)


    print(csv_paths_by_cluster_id)


    n_part = "all" if n_mentions is None else str(n_mentions)
    ocean_folder = Path("./outputs") / "OCEAN_profiles" / n_part

    doc = annotate_doc_with_ocean_folder(
        doc,
        ocean_folder,
    )
    save_doc(doc, OCEAN_SCORED_DOC_PATH)
    print("\n\n\n[OCEAN scoring] Saved annotated Doc to:", OCEAN_SCORED_DOC_PATH)

ocean = require_ocean_layer(doc)
print(ocean.summary())
#### test

In [ ]:
coref = doc._.coref_layer
ocean = doc._.ocean_layer

for cluster_id in cluster_ids:
    print(coref.clusters[cluster_id].canonical_name)
    print(ocean.cluster(cluster_id).scores.as_dict())

## Edge extraction

In [ ]:
for relation in relation_catalog:
    print(relation)

In [ ]:
def print_tree(g, root=None, indent=""):
    def label(n):
        data = g.nodes[n]
        return data.get("human_readable_label") or data.get("label") or str(n).split("#")[-1]

    if root is None:
        roots = [n for n in g.nodes if g.in_degree(n) == 0]
        for r in roots:
            print_tree(g, r)
        return

    print(indent + label(root))

    children = sorted(g.successors(root), key=label)
    for child in children:
        print_tree(g, child, indent + "    ")

print_tree(ontology_graph)

In [ ]:
# Inspect relation_router

import pandas as pd

def short(iri):
    iri = str(iri)
    return iri.rsplit("#", 1)[-1] if "#" in iri else iri.rstrip("/").rsplit("/", 1)[-1]

catalog = relation_router.relation_catalog

router_df = pd.DataFrame([
    {
        "property": short(spec.iri),
        "domains": ", ".join(short(x) for x in spec.domains),
        "ranges": ", ".join(short(x) for x in spec.ranges),
        "label": spec.human_readable_label,
        "description": spec.description,
    }
    for spec in catalog.values()
])

print(f"Object properties in router: {len(router_df)}")
display(router_df.sort_values("property"))

In [ ]:
relation_candidates_df = extract_relation_candidates(
    doc,
    cluster_type_layer=doc._.ontology_layer,
)

relation_candidates_df[
    [
        "source_cluster_id",
        "source_canonical_name",
        "source_class_iri",
        "predicate",
        "target_cluster_id",
        "target_canonical_name",
        "target_class_iri",
        "sentence_text",
        "is_passive",
        "is_negated",
    ]
]

export_routed_relation_candidates_jsonl(
    doc=doc,
    cluster_type_layer=doc._.ontology_layer,
    relation_router=relation_router,
    output_path=ROUTED_RELATION_CANDIDATES_PATH,
    print_discards=True,
    overwrite=True,
)

In [ ]:
selector = load_relation_nli_selector(
    RelationNLIConfig(
        pair_batch_size=RELATION_PAIR_BATCH_SIZE,
    )
)

export_relation_assignments_csv(
    input_path=ROUTED_RELATION_CANDIDATES_PATH,
    output_path=RELATION_ASSIGNMENTS_PATH,
    selector=selector,
    overwrite=RELATION_OVERWRITE_STAGE_2,
    resume=RELATION_RESUME_STAGE_2,
)

In [ ]:
export_cluster_assertions_csv(
    assignments_path=RELATION_ASSIGNMENTS_PATH,
    output_path=CLUSTER_ASSERTIONS_PATH,
    aggregation_config=RelationAggregationConfig(
        aggregation_method="sum_softmax_by_cluster_pair",
        min_support_count=1,
        min_score=0.0,
    ),
    overwrite=RELATION_OVERWRITE_STAGE_3,
)

In [ ]:
relation_layer = annotate_relation_layer_from_files(
    doc=doc,
    assignments_path=RELATION_ASSIGNMENTS_PATH,
    cluster_assertions_path=CLUSTER_ASSERTIONS_PATH,
    force=True,
)

print(relation_layer.summary())

In [ ]:
coref = require_coref_layer(doc)
relations = require_relation_layer(doc)

for assignment_id in list(relations.assignments)[:20]:
    source = relations.source_cluster(coref, assignment_id)
    target = relations.target_cluster(coref, assignment_id)
    predicate = relations.predicate_span(doc, assignment_id)
    prop = relations.chosen_object_property_iri(assignment_id)
    conf = relations.confidence(assignment_id)

    print(
        f"{source.canonical_name} -- {predicate.text} / {prop} "
        f"({conf:.3f}) --> {target.canonical_name}"
    )


In [ ]:
save_doc(doc, RELATION_DOC_PATH)